In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             r2_score, accuracy_score, precision_score,
                             recall_score, f1_score, classification_report)
from sklearn.preprocessing import LabelEncoder
import joblib
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

Libraries loaded.


In [8]:
TARGET_BARANGAYS = ['Nangka', 'Tumana', 'Malanday']


In [2]:
# Load processed dataset
df = pd.read_csv('../data/processed_dataset.csv')
df['week_start'] = pd.to_datetime(df['week_start'])

print(f'Dataset shape: {df.shape}')
print(df.head())
print('\nData types:')
print(df.dtypes)

Dataset shape: (657, 7)
  barangay week_start  case_count  outbreak  rainfall_lag   temp_lag  \
0   Nangka 2022-01-24           0         0           0.0  23.800000   
1   Nangka 2022-01-31           0         0           7.6  24.557143   
2   Nangka 2022-02-07           0         0           3.4  24.371429   
3   Nangka 2022-02-14           0         0          18.1  24.957143   
4   Nangka 2022-02-21           0         0          27.6  26.242857   

   humidity_lag  
0     77.000000  
1     76.714286  
2     76.000000  
3     76.285714  
4     79.571429  

Data types:
barangay                   str
week_start      datetime64[us]
case_count               int64
outbreak                 int64
rainfall_lag           float64
temp_lag               float64
humidity_lag           float64
dtype: object


In [3]:
# Encode barangay as numeric
le = LabelEncoder()
df['barangay_encoded'] = le.fit_transform(df['barangay'])

print('Barangay encoding:')
for i, name in enumerate(le.classes_):
    print(f'  {name} → {i}')

# Define features and targets
FEATURES = ['barangay_encoded', 'rainfall_lag', 'temp_lag', 'humidity_lag']

X = df[FEATURES]
y_reg = df['case_count']   # Random Forest target
y_clf = df['outbreak']     # Logistic Regression target

print(f'\nFeatures shape: {X.shape}')
print(f'Regression target shape: {y_reg.shape}')
print(f'Classification target shape: {y_clf.shape}')

Barangay encoding:
  Malanday → 0
  Nangka → 1
  Tumana → 2

Features shape: (657, 4)
Regression target shape: (657,)
Classification target shape: (657,)


In [4]:
# Train/test split — 80/20, time-aware (no shuffle)
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, shuffle=False
)

print(f'Training rows: {len(X_train)}')
print(f'Test rows:     {len(X_test)}')

Training rows: 525
Test rows:     132


In [5]:
# Train Random Forest Regressor
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train, y_reg_train)
y_pred_rf = rf_model.predict(X_test)

# Evaluate
rmse = np.sqrt(mean_squared_error(y_reg_test, y_pred_rf))
mae  = mean_absolute_error(y_reg_test, y_pred_rf)
r2   = r2_score(y_reg_test, y_pred_rf)

print('=== Random Forest Regressor ===')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

=== Random Forest Regressor ===
RMSE : 3.4375
MAE  : 2.1934
R²   : -0.3743


In [6]:
# Diagnose — check feature importance and data distribution
print('Feature importances:')
for feat, imp in zip(FEATURES, rf_model.feature_importances_):
    print(f'  {feat}: {imp:.4f}')

print('\nCase count distribution:')
print(df['case_count'].describe())

print('\nCase count by barangay:')
print(df.groupby('barangay')['case_count'].describe())

print('\nCorrelation with case_count:')
print(df[['case_count','rainfall_lag','temp_lag','humidity_lag']].corr()['case_count'])

Feature importances:
  barangay_encoded: 0.2093
  rainfall_lag: 0.2349
  temp_lag: 0.2735
  humidity_lag: 0.2823

Case count distribution:
count    657.000000
mean       1.503805
std        2.476795
min        0.000000
25%        0.000000
50%        0.000000
75%        2.000000
max       28.000000
Name: case_count, dtype: float64

Case count by barangay:
          count      mean       std  min  25%  50%  75%   max
barangay                                                     
Malanday  219.0  1.456621  2.554307  0.0  0.0  0.0  2.0  11.0
Nangka    219.0  1.739726  2.947682  0.0  0.0  1.0  2.0  28.0
Tumana    219.0  1.315068  1.775548  0.0  0.0  1.0  2.0  10.0

Correlation with case_count:
case_count      1.000000
rainfall_lag    0.194236
temp_lag       -0.115764
humidity_lag    0.284348
Name: case_count, dtype: float64


In [9]:
# Fix: add multiple lag windows and rolling mean features
lag_dfs = []

for brgy in TARGET_BARANGAYS:
    df_brgy = df[df['barangay'] == brgy].copy().sort_values('week_start')
    
    # Multiple climate lags
    for lag in [1, 2, 3, 4]:
        df_brgy[f'rainfall_lag{lag}'] = df_brgy['rainfall_lag'].shift(lag - 4) if lag < 4 else df_brgy['rainfall_lag']
        df_brgy[f'temp_lag{lag}']     = df_brgy['temp_lag'].shift(lag - 4) if lag < 4 else df_brgy['temp_lag']
        df_brgy[f'humidity_lag{lag}'] = df_brgy['humidity_lag'].shift(lag - 4) if lag < 4 else df_brgy['humidity_lag']
    
    # Rolling case count (4-week window)
    df_brgy['cases_roll4'] = df_brgy['case_count'].shift(1).rolling(4).mean()
    
    # Month as seasonal feature
    df_brgy['month'] = df_brgy['week_start'].dt.month
    
    lag_dfs.append(df_brgy)

df_enhanced = pd.concat(lag_dfs).dropna().reset_index(drop=True)

print(f'Enhanced dataset rows: {df_enhanced.shape}')
print(df_enhanced.head())

Enhanced dataset rows: (636, 22)
  barangay week_start  case_count  outbreak  rainfall_lag   temp_lag  \
0   Nangka 2022-02-21           0         0          27.6  26.242857   
1   Nangka 2022-02-28           0         0           2.7  25.671429   
2   Nangka 2022-03-07           0         0          20.3  26.342857   
3   Nangka 2022-03-14           0         0          27.6  26.385714   
4   Nangka 2022-03-21           0         0          15.5  26.614286   

   humidity_lag  barangay_encoded  rainfall_lag1  temp_lag1  ...  temp_lag2  \
0     79.571429                 1           27.6  26.385714  ...  26.342857   
1     73.428571                 1           15.5  26.614286  ...  26.385714   
2     74.714286                 1            2.7  27.414286  ...  26.614286   
3     77.428571                 1           36.3  27.428571  ...  27.414286   
4     74.285714                 1           43.5  27.714286  ...  27.428571   

   humidity_lag2  rainfall_lag3  temp_lag3  humidity_lag3  